# Teste do Conformal Prediction

In [5]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go

import time

from modelo_pacheco import *
from modelo_pacheco_paralelo import *

# Gera os dados

In [6]:
treino_val_max = 2
treino_val_min = -2

test_val_max = 2
test_val_min = -2

erro_sistematico_x = 0.5
erro_aleatorio_x = 0.5

erro_sistematico_y = 0.5
erro_aleatorio_y = 0.5

num_amostras_treino = 100
num_amostras_teste = 100

#### Modelo do sistema simuado

In [7]:
def modelo(x):
    return 2*x + x**2

#### Geração dos dados

In [8]:
X_true_train = torch.linspace(treino_val_max, treino_val_min, num_amostras_treino)
y_true_train = modelo(X_true_train)

X_measured_train = X_true_train + erro_aleatorio_x*torch.randn(num_amostras_treino) + erro_sistematico_x
y_calc_train = modelo(X_measured_train)
y_measured_train = modelo(X_measured_train) + erro_aleatorio_y*torch.randn(num_amostras_treino) + erro_sistematico_y

In [9]:
X_true_test = torch.linspace(test_val_max, test_val_min, num_amostras_teste)
y_true_test = modelo(X_true_test)

X_measured_test = X_true_test + erro_aleatorio_x*torch.randn(num_amostras_teste) + erro_sistematico_x
y_calc_test = modelo(X_measured_test)
y_measured_test = modelo(X_measured_test) + erro_aleatorio_y*torch.randn(num_amostras_teste) + erro_sistematico_y

#### Gráficos

In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=X_measured_train.numpy(), y=y_measured_train.numpy(), mode='markers', name='Measured Values', marker=dict(color='blue', size=5, opacity=0.5)))
fig.add_trace(go.Scatter(x=X_true_train.numpy(), y=y_true_train.numpy(), mode='markers', name='True Values', marker=dict(color='red', size=5, opacity=1)))
fig.update_layout(title='X vs Y de Treino', xaxis_title='X', yaxis_title='Y')
fig.show()

In [11]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=X_measured_test.numpy(), y=y_measured_test.numpy(), mode='markers', name='Measured Values', marker=dict(color='blue', size=5, opacity=0.5)))
fig.add_trace(go.Scatter(x=X_true_test.numpy(), y=y_true_test.numpy(), mode='markers', name='True Values', marker=dict(color='red', size=5, opacity=1)))
fig.update_layout(title='X vs Y de Teste', xaxis_title='X', yaxis_title='Y')
fig.show()

# Modelo de Estimativa

In [ ]:
MC_simulation_samples = 1000
n_models_ensemble = 1000

### Modelo com For

#### Treinamento

In [ ]:
start_time = time.time()

# Definição do modelo
def model_fn():
    return MLP(input_dim=1, hidden_layers=[64, 32], activation="relu", dropout=0)

# Configuração
config = TrainerConfig(epochs=100, lr=1e-3)

# INFERÊNCIA
inf_ens = InferenceEnsemble(n_models=n_models_ensemble, model_fn=model_fn, trainer_config=config)
inf_ens.fit(X_measured_train.reshape((-1, 1)).numpy(), y_measured_train.numpy())

# INCERTEZA
unc_ens = UncertaintyEnsemble(n_models=n_models_ensemble, model_fn=model_fn, trainer_config=config)
unc_ens.fit(X_measured_train.reshape((-1, 1)).numpy(), y_measured_train.numpy(), 
            input_std=erro_aleatorio_x)

end_time = time.time()
print(f"Tempo do Treinamento: {end_time - start_time:.4f} segundos")

### Inferencia

In [ ]:

# Predição
y_pred = inf_ens.predict(x_new)

# Incerteza
U_I = unc_ens.predict_uncertainty(x_new, mcs_samples=MC_simulation_samples, 
                                  input_std=erro_aleatorio_x, u_M=erro_aleatorio_y)

end_time = time.time()
print(f"Tempo total: {end_time - start_time:.2f} segundos")

print("Predição:", y_pred)
print("Incerteza:", U_I)

Usando dispositivo: cuda


/home/modelo_pacheco.py:176: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:274.)
  x_tensor = torch.tensor([x], dtype=torch.float32, device=trainer.config.device)


Tempo total: 24.43 segundos
Predição: 2.4905584255854287
Incerteza: 0.5260250568389893


#### Modelo Paralelizado

In [13]:
start_time = time.time()

def model_fn():
    return nn.Sequential(
        nn.Linear(1, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    )

config = TrainerConfigParallel(epochs=100, lr=1e-3)

# Inferência
inf = InferenceEnsembleParallel(30, model_fn, config)
inf.fit(X_measured_train.reshape((-1, 1)).numpy(), y_measured_train.numpy())

# Incerteza
unc = UncertaintyEnsembleParallel(100, model_fn, config)
unc.fit(X_measured_train.reshape((-1, 1)).numpy(), y_measured_train.numpy(), input_std=0.01)

x_new = np.array([0.5])

y_pred = inf.predict(x_new)

U_I = unc.predict_uncertainty(x_new, 1000, 0.01, 0.038)

end_time = time.time()
print(f"Tempo total: {end_time - start_time:.2f} segundos")

print("Pred:", y_pred)
print("Unc:", U_I)

Usando dispositivo: cuda
Tempo total: 0.38 segundos
Pred: 2.113896608352661
Unc: 0.9424028992652893
